In [ ]:
# Autoreload of imports
%reload_ext autoreload
%autoreload 2

In [ ]:
# Imports
import os

import torch

import shutil

import numpy as np

from math import inf

from utils import create_dataset, masked_mse
from pytorch_utils import heights_to_grid_masks

from plot import make_gif, multiple_grid

In [ ]:
# VAE and LS models
# Change these to the paths of your models

vae_file = 'models/vae/glittering-tiger-21.pt'
ls_file = 'models/latentsimulator/dazzling-rabbit-24.pt'

In [ ]:
# Load VAE
vae = torch.load(vae_file, map_location='cpu')

In [ ]:
# Load LS
ls = torch.load(ls_file, map_location='cpu')

In [ ]:
# Load test dataset
dataset = create_dataset(sequence_stride=10,
                         window_size=0,
                         use_transitions=True,
                         num_simulations=inf)

grids = torch.tensor(
    dataset['test_seq_grids']).float().unsqueeze(2)

features = torch.tensor(dataset['test_seq_features']).float()

heights = torch.tensor(dataset['test_seq_heights']).float()

num_simu, max_length, _, max_height, width = grids.shape
shape = (num_simu, max_length, max_height, width)

initial_grid = torch.full((num_simu, 1, max_height, width), 20.)


In [ ]:
# Simulate
vae.eval()
ls.eval()

with torch.no_grad():
    initial_latent = vae.encode(initial_grid)
    latent_dim = initial_latent.shape[-1]

    preds = ls.simulate(features, initial_latent)

    preds = vae.decode(preds.view(-1, latent_dim))
        
masks = heights_to_grid_masks(preds.shape, heights)

preds[masks == False] = 20.

grids = grids.view(shape)
masks = masks.view(shape)
preds = preds.view(shape)


In [ ]:
# Compute MSE
n_preds = preds.numpy()
n_grids = grids.numpy()
n_heights = heights.numpy()

mse = masked_mse(n_preds, n_grids, n_heights)
print(f'MSE: {mse:.2f}')

In [ ]:
# Generate the images of the GIF
diff = torch.abs(preds - grids)

inds = heights[0] != 0.
preds = preds[0, inds].numpy()
grids = grids[0, inds].numpy()
diff = diff[0, inds].numpy()

min_sim = min([np.min(preds), np.min(grids)])
max_sim = max([np.max(preds), np.max(grids)])

min_err = np.min(diff)
max_err = np.max(diff)

os.makedirs('imgs/', exist_ok=True)
ind_im = 0
for t in range(0, preds.shape[0], 3):
    multiple_grid([grids[t], preds[t], diff[t]],
                  vmin_sim=min_sim,
                  vmax_sim=max_sim,
                  vmin_err=min_err,
                  vmax_err=max_err,
                  save=f'imgs/image_{ind_im}.png',
                  show=False)

    ind_im += 1


In [ ]:
# Create the GIF
make_gif(fp_in="imgs/image_*.png", fp_out='world_model.gif')
shutil.rmtree('imgs/')